# EDA Global Superstore (Soal 4)
**Input**: `DWH_Global_Superstore.xlsx` (hasil ETL: `Fact_Sales` + 5 dimensi)  
**Output**:
- Tabel statistik & `eda_output/*.csv`
- `eda_output/gambar3_histogram.png`
- `eda_output/gambar4_margin_diskon.png`
- `eda_output/gambar5_margin_subkategori.png`
- `eda_output/gambar6_market_tren.png`

In [1]:
import os
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

DWH = "DWH_Global_Superstore.xlsx"
OUT = "eda_output"
os.makedirs(OUT, exist_ok=True)
pd.options.display.float_format = "{:,.3f}".format
pd.options.display.width = 200

## 1. Load & Join Data Warehouse

In [1]:
def load():
    s = lambda n: pd.read_excel(DWH, sheet_name=n)
    fact = s("Fact_Sales")
    d = (fact
         .merge(s("Dim_Customer"), on="customer_key")
         .merge(s("Dim_Product"), on="product_key")
         .merge(s("Dim_Geography"), on="geography_key")
         .merge(s("Dim_Shipping"), on="shipping_key")
         .merge(s("Dim_Date")[["date_key", "year"]],
                left_on="order_date_key", right_on="date_key"))
    assert len(d) == len(fact), "join mengubah jumlah baris"
    print(f"[LOAD] {len(d)} baris | Sales={d.sales.sum():,.2f} | "
          f"Profit={d.profit.sum():,.2f}")
    return d

def margin(g):
    return g["profit"].sum() / g["sales"].sum()

df = load()

[LOAD] 250 baris | Sales=741,918.88 | Profit=139,936.89


## 2. Statistik Deskriptif & Korelasi (4.1)

In [1]:
def descriptive(d):
    cols = ["sales", "quantity", "discount", "profit", "shipping_cost",
            "delivery_days", "profit_margin"]
    t = d[cols].describe().T[["mean", "std", "min", "25%", "50%", "75%", "max"]]
    t.columns = ["Mean", "Std", "Min", "Q1", "Median", "Q3", "Max"]
    print("\n=== 4.1 Statistik deskriptif ===")
    print(t)
    t.to_csv(f"{OUT}/statistik_deskriptif.csv")

    print("\nSkewness:")
    print(d[cols].skew())

    c = d[["discount", "profit", "profit_margin", "sales", "shipping_cost"]].corr()
    pairs = [("discount", "profit"), ("discount", "profit_margin"),
             ("sales", "profit"), ("shipping_cost", "profit"),
             ("shipping_cost", "sales")]
    corr = pd.DataFrame([(a, b, c.loc[a, b]) for a, b in pairs],
                        columns=["Variabel 1", "Variabel 2", "r"])
    print("\n=== Korelasi Pearson ===")
    print(corr)
    corr.to_csv(f"{OUT}/korelasi.csv", index=False)

    neg = d[d.profit < 0]
    print(f"\nBaris profit negatif: {len(neg)} ({len(neg)/len(d):.1%}), "
          f"total rugi {neg.profit.sum():,.2f}")
    print(f"Rata-rata shipping_cost / sales per baris: "
          f"{(d.shipping_cost / d.sales).mean():.1%} | "
          f"dari total: {d.shipping_cost.sum() / d.sales.sum():.1%}")

descriptive(df)


=== 4.1 Statistik deskriptif ===
                   Mean       Std        Min        Q1    Median        Q3       Max
sales         2,967.676 1,520.838    957.600 1,904.062 2,620.972 3,620.842 9,892.740
quantity          6.596     2.656      2.000     5.000     6.000     8.000    14.000
discount          0.086     0.131      0.000     0.000     0.000     0.100     0.700
profit          559.748 1,014.577 -6,599.978   144.056   515.420   907.970 4,946.370
shipping_cost   538.379   130.447    384.410   438.925   498.450   603.352   933.570
delivery_days     2.512     1.722      0.000     1.000     2.000     4.000     7.000
profit_margin     0.179     0.258     -1.467     0.070     0.222     0.341     0.500

Skewness:
sales            1.689
quantity         0.802
discount         2.102
profit          -1.081
shipping_cost    1.153
delivery_days    0.465
profit_margin   -2.545
dtype: float64

=== Korelasi Pearson ===
      Variabel 1     Variabel 2      r
0       discount         profit -0

## 3. Ringkasan Kelompok & Frekuensi Kategorikal

In [1]:
def group_tables(d):
    order = ["Tanpa diskon", "Rendah (<=10%)", "Sedang (10-30%)", "Tinggi (>30%)"]
    disc = (d.groupby("discount_level")
            .apply(lambda g: pd.Series({"n": len(g), "sales": g.sales.sum(),
                                        "profit": g.profit.sum(),
                                        "margin": margin(g)}),
                   include_groups=False)
            .reindex(order))
    sub = (d.groupby(["category", "sub_category"])
           .apply(lambda g: pd.Series({"n": len(g), "sales": g.sales.sum(),
                                       "profit": g.profit.sum(),
                                       "margin": margin(g),
                                       "avg_discount": g.discount.mean()}),
                  include_groups=False)
           .sort_values("margin", ascending=False))
    cat = (d.groupby("category")
           .apply(lambda g: pd.Series({"n": len(g), "sales": g.sales.sum(),
                                       "profit": g.profit.sum(),
                                       "margin": margin(g),
                                       "avg_discount": g.discount.mean()}),
                  include_groups=False))
    mkt = (d.groupby("market")
           .apply(lambda g: pd.Series({"n": len(g), "sales": g.sales.sum(),
                                       "profit": g.profit.sum(),
                                       "margin": margin(g),
                                       "avg_discount": g.discount.mean()}),
                  include_groups=False)
           .sort_values("sales", ascending=False))
    yr = (d.groupby("year")
          .apply(lambda g: pd.Series({"n": len(g), "sales": g.sales.sum(),
                                      "profit": g.profit.sum(),
                                      "margin": margin(g)}),
                 include_groups=False))
    ship = (d.groupby("ship_mode")
            .apply(lambda g: pd.Series({"n": len(g),
                                        "avg_shipping_cost": g.shipping_cost.mean(),
                                        "avg_delivery_days": g.delivery_days.mean(),
                                        "margin": margin(g)}),
                   include_groups=False)
            .sort_values("avg_delivery_days"))

    for name, t in [("Diskon", disc), ("Sub-category", sub), ("Category", cat),
                    ("Market", mkt), ("Tahun", yr), ("Ship Mode", ship)]:
        print(f"\n=== Ringkasan per {name} ===")
        print(t)
        t.to_csv(f"{OUT}/ringkasan_{name.lower().replace(' ', '_').replace('-', '_')}.csv")

    print("\n=== Frekuensi variabel kategorikal ===")
    for c in ["category", "segment", "market", "ship_mode", "order_priority"]:
        print(f"\n{c}:\n{d[c].value_counts()}")
    return disc, sub, mkt, yr

tabs = group_tables(df)


=== Ringkasan per Diskon ===
                      n       sales      profit  margin
discount_level                                         
Tanpa diskon    138.000 416,447.780 122,709.383   0.295
Rendah (<=10%)   50.000 140,702.021  24,828.105   0.176
Sedang (10-30%)  46.000 133,285.676  14,777.922   0.111
Tinggi (>30%)    16.000  51,483.408 -22,378.518  -0.435

=== Ringkasan per Sub-category ===
                                  n       sales     profit  margin  avg_discount
category        sub_category                                                    
Office Supplies Binders       8.000  48,117.464 21,406.333   0.445         0.100
Technology      Accessories   7.000  15,104.310  4,200.259   0.278         0.000
Office Supplies Storage       6.000  11,518.906  3,171.840   0.275         0.017
Technology      Copiers      31.000  79,554.960 19,741.373   0.248         0.063
Office Supplies Appliances   37.000 114,314.893 27,278.625   0.239         0.073
Furniture       Bookcases    27

## 4. Visualisasi Data (4.2)

In [1]:
from IPython.display import display, HTML

def plots(d, disc, sub, mkt, yr):
    plt.rcParams.update({"font.size": 10, "axes.spines.top": False,
                         "axes.spines.right": False})

    spacer = lambda: display(HTML("<div style='height: 60px; border-bottom: 1px dashed #ddd; margin-bottom: 30px;'></div>"))

    for col, color in zip(["sales", "profit", "shipping_cost"],
                           ["#1F3864", "#0F6E56", "#BA7517"]):
        fig, a = plt.subplots(figsize=(8.5, 4.5))
        a.hist(d[col], bins=25, color=color, edgecolor="white")
        a.axvline(d[col].median(), color="black", ls="--", lw=1.2)
        a.set_title(f"Distribusi {col.replace('_', ' ').title()}", pad=15, fontsize=12, fontweight="bold")
        a.set_xlabel(f"median = {d[col].median():,.1f}", labelpad=12)
        a.set_ylabel("Frekuensi", labelpad=12)
        fig.tight_layout()
        fig.savefig(f"{OUT}/gambar3_histogram_{col}.png", dpi=200, bbox_inches="tight")
        plt.show()
        spacer()
        plt.close(fig)

    fig, a = plt.subplots(figsize=(9.5, 5.5))
    colors = ["#0F6E56" if m >= 0 else "#A32D2D" for m in disc.margin]
    bars = a.bar(disc.index, disc.margin * 100, color=colors, width=0.55)
    for b, m, n in zip(bars, disc.margin, disc.n):
        a.annotate(f"{m:.1%}\n(n={int(n)})", (b.get_x() + b.get_width() / 2, m * 100),
                   ha="center", va="bottom" if m >= 0 else "top", fontsize=9.5,
                   xytext=(0, 10 if m >= 0 else -20), textcoords="offset points")
    a.axhline(0, color="black", lw=0.8)
    a.set_ylabel("Profit margin (%)", labelpad=14)
    a.set_title("Profit margin berdasarkan tingkat diskon", fontweight="bold", pad=25, fontsize=12)
    a.margins(y=0.35)
    fig.tight_layout()
    fig.savefig(f"{OUT}/gambar4_margin_diskon.png", dpi=200, bbox_inches="tight")
    plt.show()
    spacer()
    plt.close(fig)

    s = sub.reset_index().sort_values("margin")
    fig, a = plt.subplots(figsize=(10.5, 7.5))
    colors = ["#0F6E56" if m >= 0 else "#A32D2D" for m in s.margin]
    a.barh(s.sub_category, s.margin * 100, color=colors, height=0.65)
    for i, (m, n) in enumerate(zip(s.margin, s.n)):
        a.text(m * 100 + (1.8 if m >= 0 else -1.8), i, f"{m:.1%} (n={int(n)})",
               va="center", ha="left" if m >= 0 else "right", fontsize=9)
    a.axvline(0, color="black", lw=0.8)
    a.set_xlabel("Profit margin (%)", labelpad=14)
    a.set_title("Profit margin per sub-category", fontweight="bold", pad=25, fontsize=12)
    a.margins(x=0.3)
    fig.tight_layout()
    fig.savefig(f"{OUT}/gambar5_gambar5_margin_subkategori.png", dpi=200, bbox_inches="tight")
    plt.show()
    spacer()
    plt.close(fig)

    fig, a = plt.subplots(figsize=(9.5, 6))
    x = range(len(mkt))
    w = 0.35
    a.bar([i - w / 2 for i in x], mkt.sales, w, label="Sales", color="#1F3864")
    a.bar([i + w / 2 for i in x], mkt.profit, w, label="Profit", color="#0F6E56")
    a.set_xticks(list(x))
    a.set_xticklabels(mkt.index)
    a.set_title("Sales dan Profit per Market", fontweight="bold", pad=25, fontsize=12)
    a.set_ylabel("Nilai", labelpad=14)
    a.legend(loc="upper right")
    fig.tight_layout()
    fig.savefig(f"{OUT}/gambar6a_market.png", dpi=200, bbox_inches="tight")
    plt.show()
    spacer()
    plt.close(fig)

    fig, a = plt.subplots(figsize=(9.5, 6))
    a.plot(yr.index, yr.sales, marker="o", markersize=6, color="#1F3864", label="Sales", lw=2)
    a.plot(yr.index, yr.profit, marker="o", markersize=6, color="#0F6E56", label="Profit", lw=2)
    for xi, s_, m in zip(yr.index, yr.sales, yr.margin):
        a.annotate(f"{m:.1%}", (xi, s_), textcoords="offset points",
                   xytext=(0, 12), ha="center", fontsize=9, fontweight="bold")
    a.set_xticks(list(yr.index))
    a.set_title("Tren Tahunan 2011-2014 (persen = profit margin)", fontweight="bold", pad=25, fontsize=12)
    a.set_ylabel("Nilai", labelpad=14)
    a.legend(loc="upper left")
    a.margins(y=0.2)
    fig.tight_layout()
    fig.savefig(f"{OUT}/gambar6b_tren.png", dpi=200, bbox_inches="tight")
    plt.show()
    plt.close(fig)

plots(df, *tabs)
print("EDA selesai.")